# 🎬 CogVideoX Video Generation on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdsajid-ui/Video-Making/blob/main/CogVideoX_Video_Making.ipynb)

This notebook allows you to generate high-definition AI videos using **CogVideoX-2B** and **CogVideoX-5B** models on Google Colab's free GPU (NVIDIA T4 or A100).

### Recommended Settings
- **Runtime**: `Runtime -> Change runtime type -> T4 GPU`
- **Model**: `THUDM/CogVideoX-2b` (Runs seamlessly on T4 16GB VRAM)

### Step 1: Install Dependencies

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate sentencepiece torchvision imageio imageio-ffmpeg

### Step 2: Verify GPU Acceleration

In [ ]:
!nvidia-smi

### Step 3: Load CogVideoX Pipeline

In [ ]:
import torch
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

# Load 2B model for standard Colab T4 GPU
model_id = "THUDM/CogVideoX-2b"
print(f"Loading {model_id}...")

pipe = CogVideoXPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

# Enable VRAM memory optimizations
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()

print("Model successfully loaded!")

### Step 4: Generate Video from Prompt

In [ ]:
# Set your creative prompt
prompt = "A cinematic shot of a lone traveler with a glowing lantern walking through an enchanted misty forest at dusk, ethereal fireflies, photorealistic 4k, smooth cinematic camera pan."

print(f"Generating video for: {prompt}")

video = pipe(
    prompt=prompt,
    num_videos_per_prompt=1,
    num_inference_steps=50,
    num_frames=49,
    guidance_scale=6.0,
    generator=torch.Generator(device="cuda").manual_seed(42),
).frames[0]

output_path = "output_video.mp4"
export_to_video(video, output_path, fps=8)
print(f"Video saved to {output_path}!")

### Step 5: Display Video in Colab

In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open('output_video.mp4', 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f"""
<video width=640 controls autoplay loop>
      <source src="{data_url}" type="video/mp4">
</video>
""")